In [1]:
import os
import sys
import glob
import pandas as pd

# Add src folder to python path so we can import preprocess.py
sys.path.append(os.path.abspath("../src"))
from preprocess import preprocess_dataframe

# Automatically find the downloaded CSV files
guj_csv = glob.glob("../data/gujarati/*.csv")[0]
tam_csv = glob.glob("../data/tamil/*.csv")[0]

df_guj = pd.read_csv(guj_csv)
df_tam = pd.read_csv(tam_csv)

print("Gujarati Columns:", df_guj.columns.tolist())
print("Tamil Columns:", df_tam.columns.tolist())

Gujarati Columns: ['headline', 'category']
Tamil Columns: ['news_id', 'news_date', 'news_category', 'news_title', 'news_article']


In [3]:
import os
import glob
import pandas as pd
import sys

# 1. Append src path for module access
sys.path.append(os.path.abspath("../src"))
from preprocess import clean_indic_text

# 2. Locate Kaggle CSVs
guj_files = glob.glob("../data/gujarati/*.csv")
tam_files = glob.glob("../data/tamil/*.csv")

if not guj_files or not tam_files:
    raise FileNotFoundError("Raw CSV files missing in ../data/gujarati/ or ../data/tamil/")

print(f"Found Gujarati raw file: {[os.path.basename(f) for f in guj_files]}")
print(f"Found Tamil raw file: {[os.path.basename(f) for f in tam_files]}")

# 3. Load Kaggle Raw Data
df_guj_raw = pd.read_csv(guj_files[0])
df_tam_raw = pd.read_csv(tam_files[0])

guj_text_col = 'headline' if 'headline' in df_guj_raw.columns else df_guj_raw.columns[0]
guj_label_col = 'category' if 'category' in df_guj_raw.columns else df_guj_raw.columns[1]

tam_text_col = 'news_article' if 'news_article' in df_tam_raw.columns else df_tam_raw.columns[0]
tam_label_col = 'news_category' if 'news_category' in df_tam_raw.columns else df_tam_raw.columns[1]

df_guj_clean = pd.DataFrame({'text': df_guj_raw[guj_text_col].astype(str), 'category': df_guj_raw[guj_label_col].astype(str)})
df_tam_clean = pd.DataFrame({'text': df_tam_raw[tam_text_col].astype(str), 'category': df_tam_raw[tam_label_col].astype(str)})

# 4. Standardize Categories and Clean Text
label_map = {
    '0': 'sports', '1': 'entertainment', '2': 'business',
    '0.0': 'sports', '1.0': 'entertainment', '2.0': 'business'
}
df_guj_clean['category'] = df_guj_clean['category'].str.strip().str.lower().replace(label_map)
df_tam_clean['category'] = df_tam_clean['category'].str.strip()

df_guj_clean['clean_text'] = df_guj_clean['text'].apply(clean_indic_text)
df_tam_clean['clean_text'] = df_tam_clean['text'].apply(clean_indic_text)

df_guj_clean = df_guj_clean[df_guj_clean['clean_text'].str.len() > 10].copy()
df_tam_clean = df_tam_clean[df_tam_clean['clean_text'].str.len() > 10].copy()

# 5. Class Balancing Function (Up to 2,000 per class)
def balance_and_filter_safe(df, min_samples=10, max_samples=2000):
    counts = df['category'].value_counts()
    valid_classes = counts[counts >= min_samples].index
    df_filtered = df[df['category'].isin(valid_classes)].copy()
    
    if len(df_filtered) == 0:
        return df.sample(n=min(len(df), max_samples), random_state=42).reset_index(drop=True)

    sampled_dfs = []
    for cat_name, group in df_filtered.groupby('category'):
        sample_n = min(len(group), max_samples)
        sampled_dfs.append(group.sample(n=sample_n, random_state=42))
        
    return pd.concat(sampled_dfs, ignore_index=True)

df_guj_final = balance_and_filter_safe(df_guj_clean, min_samples=10, max_samples=2000)
df_tam_final = balance_and_filter_safe(df_tam_clean, min_samples=10, max_samples=2000)

# 6. Save Processed Datasets
os.makedirs("../data/processed", exist_ok=True)
df_guj_final[['text', 'clean_text', 'category']].to_csv("../data/processed/gujarati_clean.csv", index=False)
df_tam_final[['text', 'clean_text', 'category']].to_csv("../data/processed/tamil_clean.csv", index=False)

print("\n==================================================")
print(f"✅ Gujarati Processed! Rows: {len(df_guj_final)} | Distribution:")
print(df_guj_final['category'].value_counts().to_dict())
print("-" * 50)
print(f"✅ Tamil Processed!    Rows: {len(df_tam_final)} | Categories: {df_tam_final['category'].nunique()}")
print("==================================================")

Found Gujarati raw file: ['train.csv', 'valid.csv']
Found Tamil raw file: ['tamilmurasu_dataset.csv']

✅ Gujarati Processed! Rows: 4947 | Distribution:
{'entertainment': 2000, 'business': 1840, 'tech': 1107}
--------------------------------------------------
✅ Tamil Processed!    Rows: 19897 | Categories: 15


In [4]:
# Print column names and first row of Tamil dataset
print("Tamil Columns:", df_tam.columns.tolist())
print("\nFirst 2 rows of Tamil dataset:")
print(df_tam.head(2))

Tamil Columns: ['news_id', 'news_date', 'news_category', 'news_title', 'news_article']

First 2 rows of Tamil dataset:
   news_id            news_date news_category  \
0        6  1/6/2011 2:45:49 PM        மர்மம்   
1        9  1/6/2011 2:56:51 PM        மர்மம்   

                                          news_title  \
0  தூக்கில் தொங்கும் சேவல்கள் திருடர்களை காவு வாங...   
1                பவுர்ணமி ஜாமத்தில் மாயமான கர்ப்பிணி   

                                        news_article  
0  நாலு ஆள் உயரம், முறுக்கு மீசை, கையில் வீச்சரிவ...  
1  அமானுஷ்யமான சம்பவங்கள் நம்மை சுற்றி ஆங்காங்கே ...  


In [5]:
import os
import pandas as pd

guj_path = "../data/processed/gujarati_clean.csv"
tam_path = "../data/processed/tamil_clean.csv"

if os.path.exists(guj_path) and os.path.exists(tam_path):
    df_guj = pd.read_csv(guj_path)
    df_tam = pd.read_csv(tam_path)

    print("==================================================")
    print(f"✅ GUJARATI CATEGORY DISTRIBUTION ({len(df_guj)} rows)")
    print("==================================================")
    print(df_guj['category'].value_counts().to_string())

    print("\n" + "="*50 + "\n")

    print("==================================================")
    print(f"✅ TAMIL CATEGORY DISTRIBUTION ({len(df_tam)} rows)")
    print("==================================================")
    print(df_tam['category'].value_counts().to_string())
else:
    print("❌ Files missing. Run Cell 2 first.")

✅ GUJARATI CATEGORY DISTRIBUTION (4947 rows)
category
entertainment    2000
business         1840
tech             1107


✅ TAMIL CATEGORY DISTRIBUTION (19897 rows)
category
இந்தியா              2000
உலகம்                2000
குற்றம்              2000
சினிமா(ரீல்மா)       2000
தமிழகம்              2000
மாவட்ட மசாலா         2000
விளையாட்டு           2000
ஸ்டேட் எக்ஸ்பிரஸ்    2000
தலையங்கம்            1533
வேலைவாய்ப்பு         1042
மருத்துவம்            544
ஆன்மீகம்              406
கல்வி                 238
தொழில்                 68
மர்மம்                 66
